In [ ]:
# Azure Machine Learning workspace details:
subscription = 'f920ee3b-6bdc-48c6-a487-9e0397b69322'
resource_group = 'msan-aml'

workspace = 'msan-retrieval-ranking-aml'

datastore_name = 'bingads_algo_prod_networkprotection_c08'
path = 'shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1Ranking/Data/DeepFBS_V3_20250416_20250422_20X/training_prepare/Behaviors.tsv'

# long-form Datastore uri format:
uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}/paths/{path}'

import pandas as pd
from tqdm import tqdm

reader = pd.read_csv(uri, sep='\t', 
                   names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'], 
                   chunksize=10000)
df = pd.concat(tqdm(reader, desc='Reading data in chunks (10000 rows each)'))
# df = pd.read_csv(uri, sep='\t', 
#                    names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'])

Resolving access token for scope "https://datalake.azure.net//.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration


Reading data in chunks: 542it [01:31,  5.92it/s]


In [4]:
print(df.isnull().sum())

sequence_id    0
task_id        0
label          0
events         0
demand_type    0
dtype: int64


In [5]:
df.fillna(float(0), inplace=True)

In [5]:
df.shape

(5411652, 5)

In [ ]:
def read_and_process_cosmos_data(path):
    subscription = 'f920ee3b-6bdc-48c6-a487-9e0397b69322'
    resource_group = 'msan-aml'
    workspace = 'msan-retrieval-ranking-aml'
    datastore_name = 'bingads_algo_prod_networkprotection_c08'
    
    uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}/paths/{path}'

    import pandas as pd
    from tqdm import tqdm
    chunksize = 10000  # Number of rows to read each time
    reader = pd.read_csv(uri, sep='\t', 
                    names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'], 
                    chunksize=chunksize)
    df = pd.concat(tqdm(reader, desc=f'Reading data in chunks ({chunksize} rows each)'))
    # df = pd.read_csv(uri, sep='\t', 
    #                 names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'])
    
    import numpy as np
    from datetime import datetime

    # Handle missing values
    df.dropna(inplace=True)
    
    # Convert data types
    print("Converting data types...")
    df['demand_type'] = df['demand_type'].astype(int)

    print("Flipping events...")
    # Process events - convert to integer list and reverse
    df['events'] = df['events'].apply(lambda x: [int(i) for i in x.split()][::-1])
    
    # Rearrange and rename columns
    print("Rearranging and renaming columns...")
    df = df[['sequence_id', 'events', 'label', 'task_id', 'demand_type']]
    df.rename(columns={'sequence_id': 'seq_id', 
                       'events': 'event_id', 
                       'label': 'action_weights',
                       'task_id': 'task_id', 
                       'demand_type': 'demand_type'}, inplace=True)
    
    # Generate Unix timestamps
    def generate_custom_unix_timestamps(sequence_length):
        start_date = datetime(2025, 4, 16, 0, 0, 0)
        end_date = datetime(2025, 4, 22, 23, 59, 59)
        start_timestamp = int(start_date.timestamp())
        end_timestamp = int(end_date.timestamp())
        
        if sequence_length == 1:
            return [start_timestamp]
        
        total_time_span = end_timestamp - start_timestamp
        max_interval = total_time_span // (sequence_length - 1)
        scale = min(3600, max_interval // 2)
        intervals = np.random.exponential(scale, sequence_length-1)
        intervals = np.minimum(intervals, max_interval * 0.8)
        
        total_intervals = np.sum(intervals)
        if total_intervals > total_time_span * 0.9:
            scale_factor = (total_time_span * 0.9) / total_intervals
            intervals = intervals * scale_factor
        
        timestamps = [start_timestamp]
        current_time = start_timestamp
        for interval in intervals:
            current_time += int(interval)
            current_time = min(current_time, end_timestamp)
            timestamps.append(current_time)
        return timestamps
    
    # Generate timestamps
    print("Generating timestamps...")
    timestamps_list = []
    for i, row in df.iterrows():
        seq_length = len(row['event_id'])
        timestamps = generate_custom_unix_timestamps(seq_length)
        timestamps_list.append(timestamps)
    
    df['timestamps_unix'] = timestamps_list
    
    # Generate constant timestamps as alternative
    df['timestamps_constant'] = df['event_id'].apply(lambda x: [1] * len(x))
    
    # Generate action weights
    print("Generating action weights...")
    action_weights_list = []
    for i, row in df.iterrows():
        seq_length = len(row['event_id']) - 1
        target_label = row['action_weights']
        action_weights = [0] * seq_length + [target_label]
        action_weights_list.append(action_weights)
    
    df['action_weights'] = action_weights_list
    
    # Final column ordering
    print("Finalizing DataFrame structure.")
    df = df[['seq_id', 'event_id', 'action_weights', 'timestamps_unix', 'timestamps_constant', 'task_id', 'demand_type']]
    print("Data processing complete!")
    
    return df

In [ ]:
path = 'shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1Ranking/Data/DeepFBS_V3_20250416_20250422_20X/training_prepare/Behaviors.tsv'
df = read_and_process_cosmos_data(path)
# 5.42 million rows
# 21 min！！


Reading data in chunks (10000 rows each): 542it [01:31,  5.93it/s]


Converting data types...
Flipping events...
Rearranging and renaming columns...
Generating timestamps...
Generating action weights...
Finalizing DataFrame structure...


In [6]:
subscription = 'f920ee3b-6bdc-48c6-a487-9e0397b69322'
resource_group = 'msan-aml'
workspace = 'msan-retrieval-ranking-aml'
datastore_name = 'bingads_algo_prod_networkprotection_c08'

path = 'shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1Ranking/Data/DeepFBS_V3_20250416_20250422_20X/training_prepare/Behaviors.tsv'
uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}/paths/{path}'


In [ ]:
def read_and_process_cosmos_data_partial(path, num_lines=None):
    subscription = 'f920ee3b-6bdc-48c6-a487-9e0397b69322'
    resource_group = 'msan-aml'
    workspace = 'msan-retrieval-ranking-aml'
    datastore_name = 'bingads_algo_prod_networkprotection_c08'
    
    uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}/paths/{path}'

    import pandas as pd
    from tqdm import tqdm
    print(f"Reading up to {num_lines if num_lines else 'all'} lines...")
    if num_lines:
        df = pd.read_csv(uri, sep='\t', 
                         names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'], 
                         nrows=num_lines)
    else:
        df = pd.read_csv(uri, sep='\t', 
                         names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'])
    # chunksize = 10000  # Number of rows to read each time
    # reader = pd.read_csv(uri, sep='\t', 
    #                 names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'], 
    #                 chunksize=chunksize)
    # df = pd.concat(tqdm(reader, desc=f'Reading data in chunks ({chunksize} rows each)'))
    
    import numpy as np
    from datetime import datetime

    # Handle missing values
    df.dropna(inplace=True)
    
    # Convert data types
    print("Converting data types...")
    df['demand_type'] = df['demand_type'].astype(int)

    print("Flipping events...")
    # Process events - convert to integer list and reverse
    df['events'] = df['events'].apply(lambda x: [int(i) for i in x.split()][::-1])
    
    # Rearrange and rename columns
    print("Rearranging and renaming columns...")
    df = df[['sequence_id', 'events', 'label', 'task_id', 'demand_type']]
    df.rename(columns={'sequence_id': 'seq_id', 
                       'events': 'event_id', 
                       'label': 'action_weights',
                       'task_id': 'task_id', 
                       'demand_type': 'demand_type'}, inplace=True)
    
    # Generate Unix timestamps
    def generate_custom_unix_timestamps(sequence_length):
        start_date = datetime(2025, 4, 16, 0, 0, 0)
        end_date = datetime(2025, 4, 22, 23, 59, 59)
        start_timestamp = int(start_date.timestamp())
        end_timestamp = int(end_date.timestamp())
        
        if sequence_length == 1:
            return [start_timestamp]
        
        total_time_span = end_timestamp - start_timestamp
        max_interval = total_time_span // (sequence_length - 1)
        scale = min(3600, max_interval // 2)
        intervals = np.random.exponential(scale, sequence_length-1)
        intervals = np.minimum(intervals, max_interval * 0.8)
        
        total_intervals = np.sum(intervals)
        if total_intervals > total_time_span * 0.9:
            scale_factor = (total_time_span * 0.9) / total_intervals
            intervals = intervals * scale_factor
        
        timestamps = [start_timestamp]
        current_time = start_timestamp
        for interval in intervals:
            current_time += int(interval)
            current_time = min(current_time, end_timestamp)
            timestamps.append(current_time)
        return timestamps
    
    # Generate timestamps
    print("Generating timestamps...")
    timestamps_list = []
    for i, row in df.iterrows():
        seq_length = len(row['event_id'])
        timestamps = generate_custom_unix_timestamps(seq_length)
        timestamps_list.append(timestamps)
    
    df['timestamps_unix'] = timestamps_list
    
    # Generate constant timestamps as alternative
    df['timestamps_constant'] = df['event_id'].apply(lambda x: [1] * len(x))
    
    # Generate action weights
    print("Generating action weights...")
    action_weights_list = []
    for i, row in df.iterrows():
        seq_length = len(row['event_id']) - 1
        target_label = row['action_weights']
        action_weights = [0] * seq_length + [target_label]
        action_weights_list.append(action_weights)
    
    df['action_weights'] = action_weights_list
    
    # Final column ordering
    print("Finalizing DataFrame structure.")
    df = df[['seq_id', 'event_id', 'action_weights', 'timestamps_unix', 'timestamps_constant', 'task_id', 'demand_type']]
    print("Data processing complete!")
    
    return df

In [8]:
path = 'shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1Ranking/Data/DeepFBS_V3_20250416_20250422_20X/training_prepare/Behaviors.tsv'
df = read_and_process_cosmos_data_partial(path, num_lines=50000)

Reading up to 50000 lines...
Resolving access token for scope "https://datalake.azure.net//.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
Resolving access token for scope "https://datalake.azure.net//.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
Resolving access token for scope "https://datalake.azure.net//.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
Resolving access token for scope "https://datalake.azure.net//.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
Resolving access token for scope "https://datalake.azure.net//.default" using identity of type "MANAGED

In [ ]:
def read_and_process_cosmos_data_parallel(path, num_processes=None, use_gpu=True):
    """
    Data reading and processing function with multiprocessing/multi-GPU parallel support
    
    Args:
        path: Data file path
        num_processes: Number of parallel processes, defaults to CPU core count
        use_gpu: Whether to use GPU acceleration
    """
    import multiprocessing as mp
    import numpy as np
    import pandas as pd
    from tqdm import tqdm
    from datetime import datetime
    from functools import partial
    
    # Determine number of processes
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    print(f"Using {num_processes} processes for parallel processing")
    
    # Read data
    subscription = 'f920ee3b-6bdc-48c6-a487-9e0397b69322'
    resource_group = 'msan-aml'
    workspace = 'msan-retrieval-ranking-aml'
    datastore_name = 'bingads_algo_prod_networkprotection_c08'
    
    uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}/paths/{path}'
    
    print("Reading data...")
    chunksize = 10000
    reader = pd.read_csv(uri, sep='\t', 
                    names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'], 
                    chunksize=chunksize)
    df = pd.concat(tqdm(reader, desc=f'Reading data in chunks ({chunksize} rows each)'))
    
    # Handle missing values
    df.dropna(inplace=True)
    
    # Convert data types
    print("Converting data types...")
    df['demand_type'] = df['demand_type'].astype(int)
    
    # Rearrange and rename columns
    print("Rearranging and renaming columns...")
    df = df[['sequence_id', 'events', 'label', 'task_id', 'demand_type']]
    df.rename(columns={'sequence_id': 'seq_id', 
                       'events': 'event_id', 
                       'label': 'action_weights',
                       'task_id': 'task_id', 
                       'demand_type': 'demand_type'}, inplace=True)
    
    # Split DataFrame into multiple chunks for parallel processing
    print("Splitting data for parallel processing...")
    chunk_size = len(df) // num_processes
    chunks = [df.iloc[i:i+chunk_size] for i in range(0, len(df), chunk_size)]
    
    # Define parallel processing function
    def process_chunk(chunk_data):
        """Process a single data chunk"""
        chunk_df = chunk_data.copy()
        
        # Process events - convert to integer list and reverse
        chunk_df['event_id'] = chunk_df['event_id'].apply(
            lambda x: [int(i) for i in x.split()][::-1]
        )
        
        # Generate timestamps
        def generate_custom_unix_timestamps(sequence_length):
            start_date = datetime(2025, 4, 16, 0, 0, 0)
            end_date = datetime(2025, 4, 22, 23, 59, 59)
            start_timestamp = int(start_date.timestamp())
            end_timestamp = int(end_date.timestamp())
            
            if sequence_length == 1:
                return [start_timestamp]
            
            total_time_span = end_timestamp - start_timestamp
            max_interval = total_time_span // (sequence_length - 1)
            scale = min(3600, max_interval // 2)
            intervals = np.random.exponential(scale, sequence_length-1)
            intervals = np.minimum(intervals, max_interval * 0.8)
            
            total_intervals = np.sum(intervals)
            if total_intervals > total_time_span * 0.9:
                scale_factor = (total_time_span * 0.9) / total_intervals
                intervals = intervals * scale_factor
            
            timestamps = [start_timestamp]
            current_time = start_timestamp
            for interval in intervals:
                current_time += int(interval)
                current_time = min(current_time, end_timestamp)
                timestamps.append(current_time)
            return timestamps
        
        # Generate timestamp list
        timestamps_list = []
        for _, row in chunk_df.iterrows():
            seq_length = len(row['event_id'])
            timestamps = generate_custom_unix_timestamps(seq_length)
            timestamps_list.append(timestamps)
        
        chunk_df['timestamps_unix'] = timestamps_list
        
        # Generate constant timestamps
        chunk_df['timestamps_constant'] = chunk_df['event_id'].apply(lambda x: [1] * len(x))
        
        # Generate action weights
        action_weights_list = []
        for _, row in chunk_df.iterrows():
            seq_length = len(row['event_id']) - 1
            target_label = row['action_weights']
            action_weights = [0] * seq_length + [target_label]
            action_weights_list.append(action_weights)
        
        chunk_df['action_weights'] = action_weights_list
        
        return chunk_df
    
    # Use multiprocessing
    print("Processing chunks in parallel...")
    with mp.Pool(processes=num_processes) as pool:
        processed_chunks = list(tqdm(
            pool.imap(process_chunk, chunks),
            total=len(chunks),
            desc="Processing chunks"
        ))
    
    # Merge results
    print("Merging results...")
    df_final = pd.concat(processed_chunks, ignore_index=True)
    
    # Final column ordering
    print("Finalizing DataFrame structure...")
    df_final = df_final[['seq_id', 'event_id', 'action_weights', 'timestamps_unix', 'timestamps_constant', 'task_id', 'demand_type']]
    
    print("Parallel data processing complete!")
    return df_final

In [ ]:
def read_and_process_cosmos_data_gpu(path, use_multiple_gpus=True):
    """
    GPU-accelerated data processing function (requires installation of cudf, dask-cudf)
    
    Args:
        path: Data file path
        use_multiple_gpus: Whether to use multiple GPUs
    """
    try:
        import cudf
        import dask_cudf
        import dask.dataframe as dd
        from dask.distributed import Client
        import numpy as np
        from datetime import datetime
        import torch
        
        print("GPU libraries loaded successfully")
        
        # Check available GPU count
        if torch.cuda.is_available():
            num_gpus = torch.cuda.device_count()
            print(f"Found {num_gpus} GPU(s)")
        else:
            print("No GPU available, falling back to CPU")
            return read_and_process_cosmos_data_parallel(path)
        
        # Start Dask client (multi-GPU)
        if use_multiple_gpus and num_gpus > 1:
            client = Client(n_workers=num_gpus, threads_per_worker=2, memory_limit='8GB')
            print(f"Using {num_gpus} GPUs with Dask")
        else:
            print("Using single GPU")
            
    except ImportError as e:
        print(f"GPU libraries not available: {e}")
        print("Falling back to CPU parallel processing")
        return read_and_process_cosmos_data_parallel(path)
    
    # Data source configuration
    subscription = 'f920ee3b-6bdc-48c6-a487-9e0397b69322'
    resource_group = 'msan-aml'
    workspace = 'msan-retrieval-ranking-aml'
    datastore_name = 'bingads_algo_prod_networkprotection_c08'
    
    uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}/paths/{path}'
    
    try:
        # Read data to GPU DataFrame
        print("Reading data to GPU...")
        if use_multiple_gpus and num_gpus > 1:
            # Use Dask multi-GPU reading
            df = dd.read_csv(uri, sep='\t', 
                           names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'],
                           blocksize="100MB")  # Split data into blocks
            df = df.to_backend('cudf')  # Convert to GPU backend
        else:
            # First read with pandas, then convert to cudf
            import pandas as pd
            from tqdm import tqdm
            
            chunksize = 10000
            reader = pd.read_csv(uri, sep='\t', 
                            names=['sequence_id', 'task_id', 'label', 'events', 'demand_type'], 
                            chunksize=chunksize)
            df_pandas = pd.concat(tqdm(reader, desc='Reading data in chunks'))
            df = cudf.from_pandas(df_pandas)  # Convert to GPU DataFrame
        
        print("Data loaded to GPU successfully")
        
        # Data processing on GPU
        print("Processing data on GPU...")
        
        # Handle missing values
        df = df.dropna()
        
        # Data type conversion
        df['demand_type'] = df['demand_type'].astype('int32')
        
        # Rearrange and rename columns
        df = df[['sequence_id', 'events', 'label', 'task_id', 'demand_type']]
        df = df.rename(columns={
            'sequence_id': 'seq_id',
            'events': 'event_id', 
            'label': 'action_weights',
            'task_id': 'task_id',
            'demand_type': 'demand_type'
        })
        
        # Convert result back to pandas for complex processing
        print("Converting back to pandas for complex processing...")
        if use_multiple_gpus and num_gpus > 1:
            df_pandas = df.compute().to_pandas()
        else:
            df_pandas = df.to_pandas()
        
        # Process events field - using vectorized operations
        print("Processing events field...")
        df_pandas['event_id'] = df_pandas['event_id'].str.split().apply(
            lambda x: [int(i) for i in x][::-1] if x else []
        )
        
        # Generate timestamps - batch processing
        print("Generating timestamps...")
        def batch_generate_timestamps(event_ids):
            """Batch generate timestamps"""
            start_date = datetime(2025, 4, 16, 0, 0, 0)
            end_date = datetime(2025, 4, 22, 23, 59, 59)
            start_timestamp = int(start_date.timestamp())
            end_timestamp = int(end_date.timestamp())
            
            timestamps_batch = []
            for event_id_list in event_ids:
                seq_length = len(event_id_list)
                if seq_length == 1:
                    timestamps_batch.append([start_timestamp])
                    continue
                
                total_time_span = end_timestamp - start_timestamp
                max_interval = total_time_span // (seq_length - 1)
                scale = min(3600, max_interval // 2)
                intervals = np.random.exponential(scale, seq_length-1)
                intervals = np.minimum(intervals, max_interval * 0.8)
                
                total_intervals = np.sum(intervals)
                if total_intervals > total_time_span * 0.9:
                    scale_factor = (total_time_span * 0.9) / total_intervals
                    intervals = intervals * scale_factor
                
                timestamps = [start_timestamp]
                current_time = start_timestamp
                for interval in intervals:
                    current_time += int(interval)
                    current_time = min(current_time, end_timestamp)
                    timestamps.append(current_time)
                timestamps_batch.append(timestamps)
            
            return timestamps_batch
        
        # Batch process timestamps
        df_pandas['timestamps_unix'] = batch_generate_timestamps(df_pandas['event_id'].tolist())
        
        # Generate other fields
        print("Generating other fields...")
        df_pandas['timestamps_constant'] = df_pandas['event_id'].apply(lambda x: [1] * len(x))
        
        # Generate action weights
        def batch_generate_action_weights(event_ids, labels):
            """Batch generate action weights"""
            action_weights_batch = []
            for event_id_list, label in zip(event_ids, labels):
                seq_length = len(event_id_list) - 1
                action_weights = [0] * seq_length + [label]
                action_weights_batch.append(action_weights)
            return action_weights_batch
        
        df_pandas['action_weights'] = batch_generate_action_weights(
            df_pandas['event_id'].tolist(),
            df_pandas['action_weights'].tolist()
        )
        
        # Final column ordering
        print("Finalizing DataFrame structure...")
        df_final = df_pandas[['seq_id', 'event_id', 'action_weights', 'timestamps_unix', 'timestamps_constant', 'task_id', 'demand_type']]
        
        print("GPU-accelerated data processing complete!")
        
        # Clean up resources
        if use_multiple_gpus and num_gpus > 1:
            client.close()
        
        return df_final
        
    except Exception as e:
        print(f"GPU processing failed: {e}")
        print("Falling back to CPU parallel processing")
        if 'client' in locals():
            client.close()
        return read_and_process_cosmos_data_parallel(path)

In [ ]:
# Usage examples and performance comparison

def benchmark_processing_methods(path, small_sample_size=1000):
    """
    Compare performance of different processing methods
    """
    import time
    
    print("=" * 60)
    print("PERFORMANCE BENCHMARK")
    print("=" * 60)
    
    results = {}
    
    # 1. Original single-threaded method
    print("\n1. Testing original single-threaded method...")
    start_time = time.time()
    try:
        df_original = read_and_process_cosmos_data(path)
        # Take only samples for testing
        if len(df_original) > small_sample_size:
            df_original = df_original.head(small_sample_size)
        end_time = time.time()
        results['Original (Single-threaded)'] = {
            'time': end_time - start_time,
            'rows': len(df_original),
            'status': 'Success'
        }
    except Exception as e:
        results['Original (Single-threaded)'] = {
            'time': 0,
            'rows': 0,
            'status': f'Failed: {str(e)}'
        }
    
    # 2. Multi-process parallel method
    print("\n2. Testing multi-process parallel method...")
    start_time = time.time()
    try:
        df_parallel = read_and_process_cosmos_data_parallel(path, num_processes=4)
        if len(df_parallel) > small_sample_size:
            df_parallel = df_parallel.head(small_sample_size)
        end_time = time.time()
        results['Multi-process Parallel'] = {
            'time': end_time - start_time,
            'rows': len(df_parallel),
            'status': 'Success'
        }
    except Exception as e:
        results['Multi-process Parallel'] = {
            'time': 0,
            'rows': 0,
            'status': f'Failed: {str(e)}'
        }
    
    # 3. GPU acceleration method
    print("\n3. Testing GPU-accelerated method...")
    start_time = time.time()
    try:
        df_gpu = read_and_process_cosmos_data_gpu(path, use_multiple_gpus=True)
        if len(df_gpu) > small_sample_size:
            df_gpu = df_gpu.head(small_sample_size)
        end_time = time.time()
        results['GPU-accelerated'] = {
            'time': end_time - start_time,
            'rows': len(df_gpu),
            'status': 'Success'
        }
    except Exception as e:
        results['GPU-accelerated'] = {
            'time': 0,
            'rows': 0,
            'status': f'Failed: {str(e)}'
        }
    
    # Display results
    print("\n" + "=" * 60)
    print("BENCHMARK RESULTS")
    print("=" * 60)
    
    for method, result in results.items():
        print(f"\n{method}:")
        print(f"  Status: {result['status']}")
        if result['status'] == 'Success':
            print(f"  Time: {result['time']:.2f} seconds")
            print(f"  Rows processed: {result['rows']:,}")
            if result['time'] > 0:
                print(f"  Throughput: {result['rows']/result['time']:.2f} rows/second")
    
    return results

# Recommended usage method
def process_large_dataset(path, method='auto'):
    """
    Automatically select the best processing method based on data size and hardware configuration
    
    Args:
        path: Data file path
        method: 'auto', 'single', 'parallel', 'gpu'
    """
    import psutil
    import torch
    
    if method == 'auto':
        # Check hardware configuration
        cpu_count = psutil.cpu_count()
        memory_gb = psutil.virtual_memory().total / (1024**3)
        has_gpu = torch.cuda.is_available()
        gpu_count = torch.cuda.device_count() if has_gpu else 0
        
        print(f"Hardware detected:")
        print(f"  CPU cores: {cpu_count}")
        print(f"  Memory: {memory_gb:.1f} GB")
        print(f"  GPUs: {gpu_count}")
        
        # Automatically select method
        if gpu_count >= 2:
            print("Recommended: Multi-GPU processing")
            return read_and_process_cosmos_data_gpu(path, use_multiple_gpus=True)
        elif gpu_count == 1:
            print("Recommended: Single GPU processing")
            return read_and_process_cosmos_data_gpu(path, use_multiple_gpus=False)
        elif cpu_count >= 4:
            print("Recommended: Multi-process parallel processing")
            return read_and_process_cosmos_data_parallel(path, num_processes=cpu_count)
        else:
            print("Recommended: Single-threaded processing")
            return read_and_process_cosmos_data(path)
    
    elif method == 'single':
        return read_and_process_cosmos_data(path)
    elif method == 'parallel':
        return read_and_process_cosmos_data_parallel(path)
    elif method == 'gpu':
        return read_and_process_cosmos_data_gpu(path)
    else:
        raise ValueError("Method must be 'auto', 'single', 'parallel', or 'gpu'")

In [ ]:
# Usage examples
"""
# Method 1: Automatically select the best processing method
path = 'shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1Ranking/Data/DeepFBS_V3_20250416_20250422_20X/training_prepare/Behaviors.tsv'
df = process_large_dataset(path, method='auto')

# Method 2: Manually specify parallel processing
df = read_and_process_cosmos_data_parallel(path, num_processes=8)

# Method 3: Use GPU acceleration
df = read_and_process_cosmos_data_gpu(path, use_multiple_gpus=True)

# Method 4: Performance benchmark testing
benchmark_results = benchmark_processing_methods(path, small_sample_size=5000)
"""

# Commands to install GPU acceleration dependencies (if needed)
"""
# Run the following commands in terminal or new cell to install GPU dependencies:

# Install CuDF (requires CUDA environment)
# conda install -c rapidsai -c nvidia -c conda-forge cudf

# Or use pip
# pip install cudf-cu11  # For CUDA 11.x
# pip install cudf-cu12  # For CUDA 12.x

# Install Dask
# pip install dask[complete] dask-cudf
"""

print("Multi-GPU parallel processing functions added!")
print("\nMain improvements:")
print("1. ✅ Multi-process parallel processing - CPU multi-core acceleration")
print("2. ✅ GPU-accelerated processing - Using CuDF and Dask")
print("3. ✅ Multi-GPU support - Automatic detection and allocation of GPUs")
print("4. ✅ Adaptive method selection - Automatically choose the best method based on hardware")
print("5. ✅ Performance benchmark testing - Compare performance of different methods")
print("\nNow can handle large-scale datasets! 🚀")